In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib

BASE_PATH = "/content/drive/MyDrive/Fraud_Project"

train = pd.read_csv(os.path.join(BASE_PATH, "E1_train.csv"))
validation = pd.read_csv(os.path.join(BASE_PATH, "E1_validation.csv"))
test = pd.read_csv(os.path.join(BASE_PATH, "E1_test.csv"))

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (18734, 23)
Validation shape: (6322, 23)
Test shape: (6185, 23)


In [ ]:
print("\nTrain fraud distribution:")
print(train["is_fraud"].value_counts())

print("\nValidation fraud distribution:")
print(validation["is_fraud"].value_counts())

print("\nTest fraud distribution:")
print(test["is_fraud"].value_counts())


Train fraud distribution:
is_fraud
0    18579
1      155
Name: count, dtype: int64

Validation fraud distribution:
is_fraud
0    6265
1      57
Name: count, dtype: int64

Test fraud distribution:
is_fraud
0    6121
1      64
Name: count, dtype: int64


In [ ]:
# ==========================================
# E3 - Financial Feature Engineering
# ==========================================

financial_raw_features = [
    "price",
    "Assets",
    "AssetsCurrent",
    "CashAndCashEquivalentsAtCarryingValue",
    "CommonStockSharesIssued",
    "CommonStockValue",
    "IncomeTaxExpenseBenefit",
    "LiabilitiesCurrent",
    "OperatingIncomeLoss",
    "StockholdersEquity",
    "NetIncomeLoss",
    "Liabilities"
]


def create_financial_features(df):
    data = df.copy()

    # -----------------------------
    # Derived financial indicators
    # -----------------------------

    data["current_ratio"] = (
        data["AssetsCurrent"] /
        data["LiabilitiesCurrent"].replace(0, np.nan)
    )

    data["cash_ratio"] = (
        data["CashAndCashEquivalentsAtCarryingValue"] /
        data["LiabilitiesCurrent"].replace(0, np.nan)
    )

    data["current_assets_ratio"] = (
        data["AssetsCurrent"] /
        data["Assets"].replace(0, np.nan)
    )

    data["liabilities_to_assets"] = (
        data["Liabilities"] /
        data["Assets"].replace(0, np.nan)
    )

    data["debt_to_equity"] = (
        data["Liabilities"] /
        data["StockholdersEquity"].replace(0, np.nan)
    )

    data["roa"] = (
        data["NetIncomeLoss"] /
        data["Assets"].replace(0, np.nan)
    )

    data["roe"] = (
        data["NetIncomeLoss"] /
        data["StockholdersEquity"].replace(0, np.nan)
    )

    data["operating_return_on_assets"] = (
        data["OperatingIncomeLoss"] /
        data["Assets"].replace(0, np.nan)
    )

    # Replace infinite values with NaN
    data = data.replace([np.inf, -np.inf], np.nan)

    # Final 20 financial features
    financial_features = financial_raw_features + [
        "current_ratio",
        "cash_ratio",
        "current_assets_ratio",
        "liabilities_to_assets",
        "debt_to_equity",
        "roa",
        "roe",
        "operating_return_on_assets"
    ]

    return data, financial_features


train_fin, financial_features = create_financial_features(train)
val_fin, _ = create_financial_features(validation)
test_fin, _ = create_financial_features(test)

print("Number of financial features:", len(financial_features))
print("\nFinancial features:")
for i, feature in enumerate(financial_features, 1):
    print(f"{i}. {feature}")

Number of financial features: 20

Financial features:
1. price
2. Assets
3. AssetsCurrent
4. CashAndCashEquivalentsAtCarryingValue
5. CommonStockSharesIssued
6. CommonStockValue
7. IncomeTaxExpenseBenefit
8. LiabilitiesCurrent
9. OperatingIncomeLoss
10. StockholdersEquity
11. NetIncomeLoss
12. Liabilities
13. current_ratio
14. cash_ratio
15. current_assets_ratio
16. liabilities_to_assets
17. debt_to_equity
18. roa
19. roe
20. operating_return_on_assets


In [ ]:
from sklearn.impute import SimpleImputer

# ------------------------------------------
# Financial feature matrices
# ------------------------------------------

X_train_fin = train_fin[financial_features].copy()
X_val_fin = val_fin[financial_features].copy()
X_test_fin = test_fin[financial_features].copy()

# ------------------------------------------
# Fit imputer ONLY on training data
# ------------------------------------------

financial_imputer = SimpleImputer(strategy="median")

X_train_fin_imp = financial_imputer.fit_transform(X_train_fin)
X_val_fin_imp = financial_imputer.transform(X_val_fin)
X_test_fin_imp = financial_imputer.transform(X_test_fin)

print("Training financial matrix:", X_train_fin_imp.shape)
print("Validation financial matrix:", X_val_fin_imp.shape)
print("Test financial matrix:", X_test_fin_imp.shape)

print("\nRemaining missing values:")
print("Train:", np.isnan(X_train_fin_imp).sum())
print("Validation:", np.isnan(X_val_fin_imp).sum())
print("Test:", np.isnan(X_test_fin_imp).sum())

Training financial matrix: (18734, 20)
Validation financial matrix: (6322, 20)
Test financial matrix: (6185, 20)

Remaining missing values:
Train: 0
Validation: 0
Test: 0


In [ ]:
# ==========================================
# E3 - Load E2 TF-IDF Vectorizer
# ==========================================

vectorizer_path = os.path.join(
    BASE_PATH,
    "E2_TFIDF_vectorizer.pkl"
)

tfidf_vectorizer = joblib.load(vectorizer_path)

print("TF-IDF vectorizer loaded successfully.")
print("Vocabulary size:", len(tfidf_vectorizer.vocabulary_))

TF-IDF vectorizer loaded successfully.
Vocabulary size: 50000


In [ ]:
print("MD&A missing values:")

print("Train:", train["mda"].isna().sum())
print("Validation:", validation["mda"].isna().sum())
print("Test:", test["mda"].isna().sum())

print("\nExample MD&A length:")
print("Train:", train["mda"].str.len().describe())

MD&A missing values:
Train: 0
Validation: 0
Test: 0

Example MD&A length:
Train: count    1.873400e+04
mean     8.194024e+04
std      5.975668e+04
min      3.000000e+02
25%      4.620525e+04
50%      7.050050e+04
75%      1.030540e+05
max      1.251239e+06
Name: mda, dtype: float64


In [ ]:
# ==========================================
# E3 - TF-IDF Transformation
# ==========================================

X_train_text = tfidf_vectorizer.transform(train["mda"])
X_val_text = tfidf_vectorizer.transform(validation["mda"])
X_test_text = tfidf_vectorizer.transform(test["mda"])

print("Train text matrix:", X_train_text.shape)
print("Validation text matrix:", X_val_text.shape)
print("Test text matrix:", X_test_text.shape)

print("\nMatrix type:", type(X_train_text))
print("Train non-zero values:", X_train_text.nnz)

Train text matrix: (18734, 50000)
Validation text matrix: (6322, 50000)
Test text matrix: (6185, 50000)

Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Train non-zero values: 58593880


In [ ]:
from scipy.sparse import csr_matrix, hstack

# Convert financial matrices to sparse format
X_train_fin_sparse = csr_matrix(X_train_fin_imp)
X_val_fin_sparse = csr_matrix(X_val_fin_imp)
X_test_fin_sparse = csr_matrix(X_test_fin_imp)

# Combine financial + MD&A features
X_train_combined = hstack(
    [X_train_fin_sparse, X_train_text],
    format="csr"
)

X_val_combined = hstack(
    [X_val_fin_sparse, X_val_text],
    format="csr"
)

X_test_combined = hstack(
    [X_test_fin_sparse, X_test_text],
    format="csr"
)

print("Combined train shape:", X_train_combined.shape)
print("Combined validation shape:", X_val_combined.shape)
print("Combined test shape:", X_test_combined.shape)

print("\nCombined matrix type:", type(X_train_combined))

Combined train shape: (18734, 50020)
Combined validation shape: (6322, 50020)
Combined test shape: (6185, 50020)

Combined matrix type: <class 'scipy.sparse._csr.csr_matrix'>


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# ==========================================
# E3 - Hybrid Logistic Regression
# ==========================================

y_train = train["is_fraud"].values
y_val = validation["is_fraud"].values
y_test = test["is_fraud"].values

e3_lr = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    solver="liblinear",
    random_state=42
)

print("Training E3 Logistic Regression...")

e3_lr.fit(X_train_combined, y_train)

print("Training completed.")

Training E3 Logistic Regression...
Training completed.


In [ ]:
# ==========================================
# E3 - Validation Probability Evaluation
# ==========================================

y_val_prob_e3_lr = e3_lr.predict_proba(X_val_combined)[:, 1]

val_roc_auc = roc_auc_score(y_val, y_val_prob_e3_lr)
val_pr_auc = average_precision_score(y_val, y_val_prob_e3_lr)

print("E3 Logistic Regression - Validation")
print("------------------------------------")
print("ROC-AUC:", val_roc_auc)
print("PR-AUC :", val_pr_auc)

print("\nProbability statistics:")
print("Min   :", y_val_prob_e3_lr.min())
print("Max   :", y_val_prob_e3_lr.max())
print("Mean  :", y_val_prob_e3_lr.mean())
print("Median:", np.median(y_val_prob_e3_lr))

print("\nFraud probability statistics:")
print(
    pd.Series(y_val_prob_e3_lr[y_val == 1]).describe()
)

print("\nNon-fraud probability statistics:")
print(
    pd.Series(y_val_prob_e3_lr[y_val == 0]).describe()
)

E3 Logistic Regression - Validation
------------------------------------
ROC-AUC: 0.6457904537881015
PR-AUC : 0.03275854719480672

Probability statistics:
Min   : 0.0
Max   : 0.9999994372509731
Mean  : 0.521005605515317
Median: 0.5066389300561059

Fraud probability statistics:
count    57.000000
mean      0.591957
std       0.165558
min       0.335363
25%       0.501941
50%       0.539353
75%       0.731535
max       0.990995
dtype: float64

Non-fraud probability statistics:
count    6265.000000
mean        0.520360
std         0.082573
min         0.000000
25%         0.500674
50%         0.506622
75%         0.518323
max         0.999999
dtype: float64


In [ ]:
# ==========================================
# E3 - Validation Threshold Analysis
# ==========================================

thresholds = [
    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
    0.40,
    0.30,
    0.20,
    0.10,
    0.05,
    0.02,
    0.01
]

threshold_results = []

for threshold in thresholds:
    y_val_pred = (y_val_prob_e3_lr >= threshold).astype(int)

    precision = precision_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    predicted_fraud = y_val_pred.sum()

    threshold_results.append({
        "threshold": threshold,
        "predicted_fraud": predicted_fraud,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.to_string(index=False))

 threshold  predicted_fraud  precision   recall       f1
      0.90               78   0.051282 0.070175 0.059259
      0.80              135   0.051852 0.122807 0.072917
      0.70              236   0.063559 0.263158 0.102389
      0.60              425   0.040000 0.298246 0.070539
      0.50             5121   0.008592 0.771930 0.016995
      0.40             6197   0.008230 0.894737 0.016310
      0.30             6254   0.009114 1.000000 0.018064
      0.20             6270   0.009091 1.000000 0.018018
      0.10             6298   0.009050 1.000000 0.017939
      0.05             6304   0.009042 1.000000 0.017922
      0.02             6312   0.009030 1.000000 0.017899
      0.01             6313   0.009029 1.000000 0.017896


In [ ]:
# ==========================================
# E3 - Final Test Evaluation
# ==========================================

E3_THRESHOLD = 0.70

# Test probabilities
y_test_prob_e3_lr = e3_lr.predict_proba(X_test_combined)[:, 1]

# Apply validation-selected threshold
y_test_pred_e3_lr = (
    y_test_prob_e3_lr >= E3_THRESHOLD
).astype(int)

# Metrics
test_precision = precision_score(
    y_test,
    y_test_pred_e3_lr,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred_e3_lr,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred_e3_lr,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_prob_e3_lr
)

test_pr_auc = average_precision_score(
    y_test,
    y_test_prob_e3_lr
)

cm = confusion_matrix(
    y_test,
    y_test_pred_e3_lr
)

print("E3 Logistic Regression - TEST RESULTS")
print("---------------------------------------")
print("Threshold :", E3_THRESHOLD)
print("Precision :", test_precision)
print("Recall    :", test_recall)
print("F1        :", test_f1)
print("ROC-AUC   :", test_roc_auc)
print("PR-AUC    :", test_pr_auc)

print("\nConfusion Matrix:")
print(cm)

print("\nPredicted fraud:", y_test_pred_e3_lr.sum())
print("Actual fraud   :", y_test.sum())

E3 Logistic Regression - TEST RESULTS
---------------------------------------
Threshold : 0.7
Precision : 0.0425531914893617
Recall    : 0.125
F1        : 0.06349206349206349
ROC-AUC   : 0.520850351249796
PR-AUC    : 0.016084662499202138

Confusion Matrix:
[[5941  180]
 [  56    8]]

Predicted fraud: 188
Actual fraud   : 64


In [ ]:
# E3 Cell 12 — Reduce TF-IDF dimensions for MLP

from sklearn.decomposition import TruncatedSVD
from scipy.sparse import hstack, csr_matrix

# Number of text components for the MLP
N_TEXT_COMPONENTS = 100

svd = TruncatedSVD(
    n_components=N_TEXT_COMPONENTS,
    random_state=42
)

# Fit ONLY on training text
X_train_text_svd = svd.fit_transform(X_train_text)

# Transform validation and test using the same fitted SVD
X_val_text_svd = svd.transform(X_val_text)
X_test_text_svd = svd.transform(X_test_text)

print("Reduced text shapes:")
print("Train:", X_train_text_svd.shape)
print("Validation:", X_val_text_svd.shape)
print("Test:", X_test_text_svd.shape)

print("\nExplained variance ratio:",
      svd.explained_variance_ratio_.sum())

Reduced text shapes:
Train: (18734, 100)
Validation: (6322, 100)
Test: (6185, 100)

Explained variance ratio: 0.19872663153217507


In [ ]:
# E3 Cell 13 — Combine financial features with reduced MD&A features

import numpy as np

# Convert financial matrices to dense arrays
X_train_fin_dense = np.asarray(X_train_fin_imp)
X_val_fin_dense = np.asarray(X_val_fin_imp)
X_test_fin_dense = np.asarray(X_test_fin_imp)

# Combine financial + SVD text features
X_train_e3_mlp = np.hstack([
    X_train_fin_dense,
    X_train_text_svd
])

X_val_e3_mlp = np.hstack([
    X_val_fin_dense,
    X_val_text_svd
])

X_test_e3_mlp = np.hstack([
    X_test_fin_dense,
    X_test_text_svd
])

print("E3 MLP input shapes:")
print("Train:", X_train_e3_mlp.shape)
print("Validation:", X_val_e3_mlp.shape)
print("Test:", X_test_e3_mlp.shape)

print("\nTotal features:", X_train_e3_mlp.shape[1])
print("Any NaN:", np.isnan(X_train_e3_mlp).any())
print("Any Inf:", np.isinf(X_train_e3_mlp).any())

E3 MLP input shapes:
Train: (18734, 120)
Validation: (6322, 120)
Test: (6185, 120)

Total features: 120
Any NaN: False
Any Inf: False


In [ ]:
# E3 Cell 14 — Scale hybrid features for MLP

from sklearn.preprocessing import StandardScaler

e3_mlp_scaler = StandardScaler()

# Fit ONLY on training data
X_train_e3_mlp_scaled = e3_mlp_scaler.fit_transform(
    X_train_e3_mlp
)

# Transform validation and test
X_val_e3_mlp_scaled = e3_mlp_scaler.transform(
    X_val_e3_mlp
)

X_test_e3_mlp_scaled = e3_mlp_scaler.transform(
    X_test_e3_mlp
)

print("Scaled shapes:")
print("Train:", X_train_e3_mlp_scaled.shape)
print("Validation:", X_val_e3_mlp_scaled.shape)
print("Test:", X_test_e3_mlp_scaled.shape)

print("\nTraining mean (first 5 features):")
print(X_train_e3_mlp_scaled[:, :5].mean(axis=0))

print("\nTraining std (first 5 features):")
print(X_train_e3_mlp_scaled[:, :5].std(axis=0))

print("\nAny NaN:", np.isnan(X_train_e3_mlp_scaled).any())
print("Any Inf:", np.isinf(X_train_e3_mlp_scaled).any())

Scaled shapes:
Train: (18734, 120)
Validation: (6322, 120)
Test: (6185, 120)

Training mean (first 5 features):
[ 2.24085271e-17  2.32975567e-17  1.45276007e-16  3.32059445e-16
 -4.25710623e-16]

Training std (first 5 features):
[1. 1. 1. 1. 1.]

Any NaN: False
Any Inf: False


In [ ]:
# E3 Cell 15 — Train Hybrid MLP

from sklearn.neural_network import MLPClassifier

e3_mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42
)

e3_mlp.fit(
    X_train_e3_mlp_scaled,
    y_train
)

print("E3 Hybrid MLP training completed.")
print("Iterations:", e3_mlp.n_iter_)
print("Final training loss:", e3_mlp.loss_)

E3 Hybrid MLP training completed.
Iterations: 35
Final training loss: 0.0009653636814733751


In [ ]:
# E3 Cell 16 — Hybrid MLP validation evaluation

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Validation probabilities
y_val_prob_e3_mlp = e3_mlp.predict_proba(
    X_val_e3_mlp_scaled
)[:, 1]

# Ranking metrics
val_roc_auc = roc_auc_score(
    y_val,
    y_val_prob_e3_mlp
)

val_pr_auc = average_precision_score(
    y_val,
    y_val_prob_e3_mlp
)

print("E3 Hybrid MLP - VALIDATION")
print("---------------------------------------")
print("ROC-AUC:", val_roc_auc)
print("PR-AUC :", val_pr_auc)

print("\nProbability statistics:")
print("Min   :", y_val_prob_e3_mlp.min())
print("Max   :", y_val_prob_e3_mlp.max())
print("Mean  :", y_val_prob_e3_mlp.mean())
print("Median:", np.median(y_val_prob_e3_mlp))

print("\nFraud probability:")
print(
    pd.Series(
        y_val_prob_e3_mlp[y_val == 1]
    ).describe()
)

print("\nNon-fraud probability:")
print(
    pd.Series(
        y_val_prob_e3_mlp[y_val == 0]
    ).describe()
)

E3 Hybrid MLP - VALIDATION
---------------------------------------
ROC-AUC: 0.6888030131193906
PR-AUC : 0.018965937297668944

Probability statistics:
Min   : 7.853508538082764e-83
Max   : 0.8455581310288302
Mean  : 0.005932188127995744
Median: 9.686061002656444e-05

Fraud probability:
count    57.000000
mean      0.013917
std       0.043884
min       0.000002
25%       0.000190
50%       0.000708
75%       0.009569
max       0.309480
dtype: float64

Non-fraud probability:
count    6.265000e+03
mean     5.859539e-03
std      3.196115e-02
min      7.853509e-83
25%      6.522017e-06
50%      9.516581e-05
75%      1.035728e-03
max      8.455581e-01
dtype: float64


In [ ]:
# E3 Cell 17 — Hybrid MLP validation threshold analysis

thresholds = [
    0.50,
    0.30,
    0.20,
    0.10,
    0.05,
    0.02,
    0.01,
    0.005,
    0.002,
    0.001,
    0.0005,
    0.0002,
    0.0001,
    0.00005,
    0.00001
]

threshold_results = []

for threshold in thresholds:

    y_val_pred = (
        y_val_prob_e3_mlp >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        y_val_pred,
        zero_division=0
    )

    predicted_fraud = y_val_pred.sum()

    threshold_results.append({
        "threshold": threshold,
        "predicted_fraud": predicted_fraud,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(threshold_results)

print("E3 Hybrid MLP - VALIDATION THRESHOLD ANALYSIS")
print("-----------------------------------------------")
display(threshold_df)

E3 Hybrid MLP - VALIDATION THRESHOLD ANALYSIS
-----------------------------------------------


,threshold,predicted_fraud,precision,recall,f1
0,0.50000,5,0.000000,0.000000,0.000000
1,0.30000,17,0.058824,0.017544,0.027027
2,0.20000,35,0.028571,0.017544,0.021739
3,0.10000,86,0.023256,0.035088,0.027972
4,0.05000,166,0.018072,0.052632,0.026906
5,0.02000,344,0.023256,0.140351,0.039900
6,0.01000,550,0.025455,0.245614,0.046129
7,0.00500,800,0.023750,0.333333,0.044341
8,0.00200,1256,0.017516,0.385965,0.033511
9,0.00100,1615,0.016718,0.473684,0.032297


In [ ]:
# E3 Cell 18 — Final Hybrid MLP test evaluation

E3_MLP_THRESHOLD = 0.01

# Test probabilities
y_test_prob_e3_mlp = e3_mlp.predict_proba(
    X_test_e3_mlp_scaled
)[:, 1]

# Apply validation-selected threshold
y_test_pred_e3_mlp = (
    y_test_prob_e3_mlp >= E3_MLP_THRESHOLD
).astype(int)

# Metrics
test_precision = precision_score(
    y_test,
    y_test_pred_e3_mlp,
    zero_division=0
)

test_recall = recall_score(
    y_test,
    y_test_pred_e3_mlp,
    zero_division=0
)

test_f1 = f1_score(
    y_test,
    y_test_pred_e3_mlp,
    zero_division=0
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_prob_e3_mlp
)

test_pr_auc = average_precision_score(
    y_test,
    y_test_prob_e3_mlp
)

cm = confusion_matrix(
    y_test,
    y_test_pred_e3_mlp
)

print("E3 Hybrid MLP - TEST RESULTS")
print("---------------------------------------")
print("Threshold :", E3_MLP_THRESHOLD)
print("Precision :", test_precision)
print("Recall    :", test_recall)
print("F1        :", test_f1)
print("ROC-AUC   :", test_roc_auc)
print("PR-AUC    :", test_pr_auc)

print("\nConfusion Matrix:")
print(cm)

print("\nPredicted fraud:", y_test_pred_e3_mlp.sum())
print("Actual fraud   :", y_test.sum())

E3 Hybrid MLP - TEST RESULTS
---------------------------------------
Threshold : 0.01
Precision : 0.025806451612903226
Recall    : 0.1875
F1        : 0.045368620037807186
ROC-AUC   : 0.6839900547296194
PR-AUC    : 0.025170247115570577

Confusion Matrix:
[[5668  453]
 [  52   12]]

Predicted fraud: 465
Actual fraud   : 64


In [ ]:
# E3 Cell 19 — Save Hybrid MLP artifacts

import os
import joblib

BASE_PATH = "/content/drive/MyDrive/Fraud_Project"

# Save trained MLP
joblib.dump(
    e3_mlp,
    os.path.join(BASE_PATH, "E3_Hybrid_MLP_model.pkl")
)

# Save SVD transformer
joblib.dump(
    svd,
    os.path.join(BASE_PATH, "E3_Hybrid_MLP_SVD.pkl")
)

# Save scaler
joblib.dump(
    e3_mlp_scaler,
    os.path.join(BASE_PATH, "E3_Hybrid_MLP_scaler.pkl")
)

# Save selected threshold
with open(
    os.path.join(BASE_PATH, "E3_Hybrid_MLP_threshold.txt"),
    "w"
) as f:
    f.write(str(E3_MLP_THRESHOLD))

print("E3 Hybrid MLP artifacts saved successfully.")
print("\nSaved files:")

print("1.", os.path.join(BASE_PATH, "E3_Hybrid_MLP_model.pkl"))
print("2.", os.path.join(BASE_PATH, "E3_Hybrid_MLP_SVD.pkl"))
print("3.", os.path.join(BASE_PATH, "E3_Hybrid_MLP_scaler.pkl"))
print("4.", os.path.join(BASE_PATH, "E3_Hybrid_MLP_threshold.txt"))

E3 Hybrid MLP artifacts saved successfully.

Saved files:
1. /content/drive/MyDrive/Fraud_Project/E3_Hybrid_MLP_model.pkl
2. /content/drive/MyDrive/Fraud_Project/E3_Hybrid_MLP_SVD.pkl
3. /content/drive/MyDrive/Fraud_Project/E3_Hybrid_MLP_scaler.pkl
4. /content/drive/MyDrive/Fraud_Project/E3_Hybrid_MLP_threshold.txt
